# All of September Across All Routes

In [1]:
import zipfile

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
from pathlib import Path

RAW = Path("../data/raw/511")
sorted(RAW.iterdir())

[PosixPath('../data/raw/511/.DS_Store'),
 PosixPath('../data/raw/511/2026-09-so'),
 PosixPath('../data/raw/511/2026-09-so.zip')]

## Import September 2026 Vehicle Records

In [2]:
zip_path = RAW / "2026-09-so.zip"
so = []

with zipfile.ZipFile(zip_path) as z:
    with z.open("stop_observations.txt") as f:
        reader = pd.read_csv(f, chunksize=1_000_000, dtype={
            "from_stop_id": str,
            "to_stop_id": str,
            "vehicle_id": str,
            "trip_id": str,
            "route_id": str,
        })
        for chunk in reader:
            so.append(chunk[chunk["agency_id"] == "SF"])

    with z.open("stops.txt") as f:
        sf_stops_info = pd.read_csv(f, dtype={"stop_id": str})

sept_2026 = pd.concat(so, ignore_index=True)
sept_2026.head()

,trip_id,trip_start_time,schedule_relationship,service_date,vehicle_id,stop_sequence,observed_arrival_time,observed_departure_time,uncertainty,dwell_time_secs,scheduled_dwell_time_secs,route_id,agency_id,direction_id,from_stop_id,to_stop_id,scheduled_arrival_time,scheduled_departure_time
0,SF:12108780_M11:20260930,NaN,0,20260930,8881,1,10:06:00,10:06:00,0,0.0,0.0,SF:9,SF,1.0,NaN,17520,10:06:00,10:06:00
1,SF:12108780_M11:20260930,NaN,0,20260930,8881,2,10:08:09,10:08:09,0,0.0,0.0,SF:9,SF,1.0,17520,13793,10:07:11,10:07:11
2,SF:12108780_M11:20260930,NaN,0,20260930,8881,3,10:12:01,10:12:01,0,0.0,0.0,SF:9,SF,1.0,13793,16352,10:09:00,10:09:00
3,SF:12108780_M11:20260930,NaN,0,20260930,8881,4,10:12:19,10:12:19,0,0.0,0.0,SF:9,SF,1.0,16352,17191,10:09:23,10:09:23
4,SF:12108780_M11:20260930,NaN,0,20260930,8881,5,10:13:46,10:13:46,0,0.0,0.0,SF:9,SF,1.0,17191,16350,10:10:29,10:10:29


## Pick out the columns we need and calculate arrival times in seconds

In [3]:
sept_2026_clean = sept_2026[["route_id", "trip_id", "vehicle_id", "service_date", "stop_sequence", "observed_arrival_time", "from_stop_id", "to_stop_id", "scheduled_arrival_time"]]

# turn the HH:MM:SS of arrival time into seconds for easy diffing
sept_2026_clean["arrival_secs"] = pd.to_timedelta(sept_2026_clean["observed_arrival_time"]).dt.total_seconds()

## Label each row with it's origin and dest stop names

In [4]:
stop_names = sf_stops_info[["stop_id", "stop_name"]]

# label each row with the actual names of the origin and dest stops
origin_labeled = (
    sept_2026_clean.merge(stop_names, left_on="from_stop_id", right_on="stop_id", how="left")
    .drop(columns="stop_id")
    .rename(columns={"stop_name": "from_stop_name"})
)
origin_and_dest_labeled = (
    origin_labeled.merge(stop_names, left_on="to_stop_id", right_on="stop_id", how="left")
    .drop(columns="stop_id")
    .rename(columns={"stop_name": "to_stop_name"})
)

# one trip on one day together, its stops in order
sept_2026_clean = origin_and_dest_labeled.sort_values(["service_date", "trip_id", "stop_sequence"])

## Fix inconsistency in 511's arrival time reporting 

### sometimes times come in as 24:MM:SS and sometimes as 00:MM:SS within the same trip which causes the calculated travel time to be absurd

### we find the latest point in the trip and then (since no route takes > 12 hrs to run) we can add 24hrs to those 

In [5]:
keys = ["service_date", "trip_id"]   # one trip on one day

# the latest time this trip reached, repeated on every row of the trip
trip_latest = sept_2026_clean.groupby(keys)["arrival_secs"].transform("max")

# a row is wrapped if it reads more than 12 hours earlier than that
wrapped = (trip_latest - sept_2026_clean["arrival_secs"]) > 12 * 3600

# add one day to those rows only; keep the original column for comparison
sept_2026_clean["arrival_secs_fixed"] = np.where(
    wrapped, sept_2026_clean["arrival_secs"] + 24 * 3600, sept_2026_clean["arrival_secs"]
)

before = sept_2026_clean.groupby(keys)["arrival_secs"].diff()
after  = sept_2026_clean.groupby(keys)["arrival_secs_fixed"].diff()

print("rows fixed:", wrapped.sum())
print("±12h jumps before:", (before.abs() > 12 * 3600).sum())
print("±12h jumps after: ", (after.abs() > 12 * 3600).sum())   # should be 0

sept_2026_clean["segment_travel_time"] = after

rows fixed: 190
±12h jumps before: 115
±12h jumps after:  0


In [6]:
print("negative travel times:", (sept_2026_clean["segment_travel_time"] < 0).sum())

negative travel times: 234751


In [7]:
by_trip = sept_2026_clean.groupby(keys)["segment_travel_time"]

sept_2026_clean["prev_seg_1_secs"] = by_trip.shift(1)   # the segment just before this one
sept_2026_clean["prev_seg_2_secs"] = by_trip.shift(2)   # the one before that

In [8]:
# 1. IDs → categories (do this BEFORE splitting, so train and valid share one category list)
for col in ["from_stop_id", "to_stop_id", "route_id"]:
    sept_2026_clean[col] = sept_2026_clean[col].astype("category")

# 2. when the bus LEFT the origin stop = this arrival minus the travel time (known before the segment starts)
sept_2026_clean["start_secs"] = sept_2026_clean["arrival_secs_fixed"] - sept_2026_clean["segment_travel_time"]
sept_2026_clean["hour"] = (sept_2026_clean["start_secs"] // 3600) % 24          # % 24 so 25:10 → hour 1

# 3. day of week from the service date: 0 = Monday ... 6 = Sunday
sept_2026_clean["day_of_week"] = pd.to_datetime(sept_2026_clean["service_date"].astype(str), format="%Y%m%d").dt.dayofweek

# 4. what the timetable allowed for this segment, same groupby-diff as the real travel time
sept_2026_clean["sched_secs"] = pd.to_timedelta(sept_2026_clean["scheduled_arrival_time"]).dt.total_seconds()
sept_2026_clean["sched_seg_secs"] = sept_2026_clean.groupby(keys)["sched_secs"].diff()

# Training and Test Splits

split across just the month of september will be:
training - 09/01-09/22
testing - 09/23-09/30
to accurately test the future dates nature of these predictions

## there will be two gradient boosted trees trained on this data
1. all routes
2. one tree per route

## Tree One
**params:**
- **s_1** - from_stop_id
- **s_2** - to_stop_id
- **r** - route_id
- **sd** - service_date
- **sat** - scheduled_arrival_time
- **oat_{-1}** - travel time spent last segment
- **oat_{-2}** - travel time spent two segments ago

In [9]:
# Tree One

features = ["from_stop_id", "to_stop_id", "route_id", "hour", "day_of_week", "sched_seg_secs", "prev_seg_1_secs", "prev_seg_2_secs"]

# rows with no target (each trip's first stop) can't be learned from
has_target = sept_2026_clean[sept_2026_clean["segment_travel_time"].notna()]

train = has_target[has_target["service_date"] <= 20260922]
valid = has_target[has_target["service_date"] > 20260922]

# training set
X_train = train[features]
y_train = train["segment_travel_time"]

# testing set
X_valid = valid[features]
y_valid = valid["segment_travel_time"]

y_train.describe()

count    7.073575e+06
mean     7.532761e+01
std      1.455883e+02
min     -7.390000e+03
25%      3.600000e+01
50%      5.800000e+01
75%      9.400000e+01
max      8.887000e+03
Name: segment_travel_time, dtype: float64

In [10]:
model = lgb.LGBMRegressor()            # "regressor" = predicts a number
model.fit(X_train, y_train)            # learn
preds = model.predict(X_valid)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.030127 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6973
[LightGBM] [Info] Number of data points in the train set: 7073575, number of used features: 8
[LightGBM] [Info] Start training from score 75.327606
